# data layout
## partitioning the table by column values

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = []

for i in range(1,1001):
    data.append((i,
                 "Laptop" if i%3==0 else "Mobile" if i%3==1 else "Tablet",
                 100 + i,
                 2023 if i%2==0 else 2024))

columns = ["id","product","price","year"]

df = spark.createDataFrame(data,columns)

df.show(10)


In [0]:
df.write \
.format("delta") \
.partitionBy("year") \
.mode("overwrite") \
.save("/Volumes/workspace/delta/data")


In [0]:
sales = spark.read.format("delta").load("/Volumes/workspace/delta/data")

sales.filter("year=2024").show()



In [0]:
sales.filter("product='Laptop'").show()


In [0]:
display(spark.sql("""
DESCRIBE DETAIL delta.`/Volumes/workspace/delta/data`
"""))


In [0]:
spark.sql("""
OPTIMIZE delta.`/Volumes/workspace/delta/data`
ZORDER BY (product)
""")


In [0]:
spark.sql("""
CREATE TABLE sales_cluster_demo
(
id INT,
product STRING,
price INT,
year INT
)
USING DELTA
CLUSTER BY (product)
""")


In [0]:
df.write.mode("append").saveAsTable("sales_cluster_demo")
